In [82]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
import pickle
import pandas as pd


In [64]:
df=pd.read_excel("/content/datasetB.xlsx")  #Label 0 represents Legitimate URL

#Label 1 represents Phishing URL

In [65]:
df.head()

,Labels,URLs
0,1,http://dbs.vote-friend.com/sg?ref=anything
1,0,https://www.reynoldstransfer.com/versa-lift-fo...
2,1,https://www.halisupportservice.com/Login.php
3,0,https://www.signets.com.br/wp-includes/wlwmani...
4,1,https://docs.google.com/document/d/e/2PACX-1vT...


In [66]:
df.duplicated(subset='URLs').sum()

np.int64(3)

In [67]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

In [68]:
from urllib.parse import urlparse
import re
import ipaddress
from utils import tld_bucket

def get_tld(domain):
    parts = domain.split('.')
    if len(parts) >= 2:
        if parts[-2] in ['co', 'ac', 'gov']:
            return '.' + parts[-2] + '.' + parts[-1]
    return '.' + parts[-1] if parts else ''

def is_ip_address(domain):
    try:
        ipaddress.ip_address(domain)
        return 1
    except:
        return 0

def extract_features(url: str):
    # 🔥 FIX: normalize URL
    url = url.strip().lower()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url

    parsed = urlparse(url)
    domain = parsed.netloc

    url_length = len(url)
    domain_length = len(domain)

    is_ip = is_ip_address(domain)

    tld = get_tld(domain)
    tld_length = len(tld)
    tld_risk = tld_bucket(tld)

    subdomains = max(len(domain.split('.')) - 2, 0)

    obfuscated_chars = url.count('%') + url.count('\\x')
    has_obfuscation = 1 if obfuscated_chars > 0 else 0
    obfuscation_ratio = obfuscated_chars / url_length if url_length else 0

    letters = sum(c.isalpha() for c in url)
    digits = sum(c.isdigit() for c in url)

    letter_ratio = letters / url_length if url_length else 0
    digit_ratio = digits / url_length if url_length else 0

    equals = url.count('=')
    qmark = url.count('?')
    ampersand = url.count('&')

    special_chars = sum(c in ['@', '-', '_', '%', '=', '?', '&'] for c in url)
    special_ratio = special_chars / url_length if url_length else 0

    is_https = 1 if parsed.scheme == 'https' else 0

    # 🔥 New features
    suspicious_words = [
        "login", "verify", "secure", "account",
        "bank", "update", "confirm", "password",
        "paypal", "signin", "alert"
    ]

    has_suspicious = int(any(word in url for word in suspicious_words))
    hyphen_count = url.count('-')

    return [
        url_length,
        domain_length,
        is_ip,
        tld_length,
        subdomains,
        has_obfuscation,
        obfuscated_chars,
        obfuscation_ratio,
        letters,
        letter_ratio,
        digits,
        digit_ratio,
        equals,
        qmark,
        ampersand,
        special_chars,
        special_ratio,
        is_https,
        tld_risk,
        has_suspicious,
        hyphen_count
    ]

In [69]:
X = df['URLs'].apply(extract_features)
X = pd.DataFrame(list(X), columns=FEATURE_ORDER) # Convert to DataFrame with feature names

y = df['Labels']

In [70]:
FEATURE_ORDER = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength',
    'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar',
    'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL',
    'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL',
    'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL',
    'IsHTTPS', 'TLD_risk',
    'HasSuspicious',        # 🔥 NEW
    'HyphenCount'           # 🔥 NEW
]

In [71]:
df.shape

(19997, 2)

In [72]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [73]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

preprocess = ColumnTransformer([
    ("num", StandardScaler(), FEATURE_ORDER)
])

In [93]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(
    preprocess,
    SVC(kernel='rbf',class_weight='balanced',probability=True)
)

In [94]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['URLLength', 'DomainLength',
                                                   'IsDomainIP', 'TLDLength',
                                                   'NoOfSubDomain',
                                                   'HasObfuscation',
                                                   'NoOfObfuscatedChar',
                                                   'ObfuscationRatio',
                                                   'NoOfLettersInURL',
                                                   'LetterRatioInURL',
                                                   'NoOfDegitsInURL',
                                                   'DegitRatioInURL',
                                                   'NoOfEqualsInURL',
                                                   'NoOfQMarkInURL',
                                                   'NoOfAmpersandInURL',
                                                   'NoOfOtherSpecialCharsInURL',
                                                   'SpacialCharRatioInURL',
                                                   'IsHTTPS', 'TLD_risk',
                                                   'HasSuspicious',
                                                   'HyphenCount'])])),
                ('svc', SVC(class_weight='balanced', probability=True))])

In [96]:
from sklearn.metrics import classification_report, confusion_matrix

y_prob = pipe.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)
#y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.90      0.92      2000
           1       0.91      0.94      0.92      2000

    accuracy                           0.92      4000
   macro avg       0.92      0.92      0.92      4000
weighted avg       0.92      0.92      0.92      4000

[[1806  194]
 [ 119 1881]]


In [97]:
# Save the trained SVM model to a file
filename = 'Phish1.pkl'
pickle.dump(pipe, open(filename, 'wb'))

print(f"SVM model successfully saved to {filename}")

SVM model successfully saved to Phish1.pkl


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['URLLength', 'DomainLength',
                                                   'IsDomainIP', 'TLDLength',
                                                   'NoOfSubDomain',
                                                   'HasObfuscation',
                                                   'NoOfObfuscatedChar',
                                                   'ObfuscationRatio',
                                                   'NoOfLettersInURL',
                                                   'LetterRatioInURL',
                                                   'NoOfDegitsInURL',
                                                   'DegitRatioInURL',
                                                   'NoOfEqualsInURL',
                                                   'NoOfQMarkInURL',
                                                   'NoOfAmpersandInURL',
                                                   'NoOfOtherSpecialCharsInURL',
                                                   'SpacialCharRatioInURL',
                                                   'IsHTTPS', 'TLD_risk',
                                                   'HasSuspicious',
                                                   'HyphenCount'])])),
                ('logisticregression',
                 LogisticRegression(class_weight='balanced'))])

              precision    recall  f1-score   support

           0       0.85      0.92      0.89      2000
           1       0.92      0.84      0.88      2000

    accuracy                           0.88      4000
   macro avg       0.88      0.88      0.88      4000
weighted avg       0.88      0.88      0.88      4000

[[1845  155]
 [ 317 1683]]
